# VoxMood: speech emotion classifier (RAVDESS, MLP)

Extracts MFCC, chroma and mel-spectrogram features from RAVDESS audio and trains a scikit-learn `MLPClassifier` on four emotions (calm, happy, fearful, disgust).

Adapted from the DataFlair speech emotion recognition tutorial. Download RAVDESS separately and place the `Actor_*` folders in `data/ravdess/`.

In [1]:
import librosa
import soundfile
import os, glob
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score

In [4]:
def extract_feature(file_name, mfcc, chroma, mel):
    with soundfile.SoundFile(file_name) as sound_file:
        X = sound_file.read(dtype="float32")
        sample_rate = sound_file.samplerate
    if X.ndim > 1:  # mix multi-channel audio down to mono
        X = X.mean(axis=1)
    features = []
    if mfcc:
        features.append(np.mean(librosa.feature.mfcc(y=X, sr=sample_rate, n_mfcc=40).T, axis=0))
    if chroma:
        stft = np.abs(librosa.stft(X))
        features.append(np.mean(librosa.feature.chroma_stft(S=stft, sr=sample_rate).T, axis=0))
    if mel:
        features.append(np.mean(librosa.feature.melspectrogram(y=X, sr=sample_rate).T, axis=0))
    return np.hstack(features)

In [10]:
#DataFlair - Emotions in the RAVDESS dataset
emotions={
  '01':'neutral',
  '02':'calm',
  '03':'happy',
  '04':'sad',
  '05':'angry',
  '06':'fearful',
  '07':'disgust',
  '08':'surprised'
}

#DataFlair - Emotions to observe
observed_emotions=['calm', 'happy', 'fearful', 'disgust']

# RAVDESS audio is not stored in this repository; place the Actor_* folders here
DATA_DIR = os.path.join("..", "data", "ravdess")


In [6]:
#DataFlair - Load the data and extract features for each sound file
def load_features():
    """Return feature vectors, emotion labels and actor ids for the observed emotions."""
    paths = sorted(glob.glob(os.path.join(DATA_DIR, "Actor_*", "*.wav")))  # sorted for a stable order
    x, y, actors = [], [], []
    for file in paths:
        file_name = os.path.basename(file)
        fields = file_name.split("-")
        emotion = emotions[fields[2]]
        if emotion not in observed_emotions:
            continue
        x.append(extract_feature(file, mfcc=True, chroma=True, mel=True))
        y.append(emotion)
        actors.append(fields[6].split(".")[0])
    if not x:
        raise FileNotFoundError(f"No matching RAVDESS .wav files found under {DATA_DIR}")
    return np.array(x), np.array(y), np.array(actors)

In [14]:
#DataFlair - Split the dataset (random split by file, as in the tutorial)
features, labels, actors = load_features()
x_train, x_test, y_train, y_test = train_test_split(features, labels, test_size=0.25, random_state=9)

KeyError: ignored

In [13]:
#DataFlair - Get the number of features extracted
print(f'Features extracted: {x_train.shape[1]}')

NameError: ignored

In [ ]:
#DataFlair - Get the shape of the training and testing datasets
print((x_train.shape[0], x_test.shape[0]))

In [ ]:
#DataFlair - Initialize the Multi Layer Perceptron Classifier
model=MLPClassifier(alpha=0.01, batch_size=256, epsilon=1e-08, hidden_layer_sizes=(300,), learning_rate='adaptive', max_iter=500, random_state=9)

In [ ]:
#DataFlair - Train the model
model.fit(x_train,y_train)

In [ ]:
#DataFlair - Predict for the test set
y_pred=model.predict(x_test)

In [ ]:
#DataFlair - Calculate the accuracy of our model
accuracy=accuracy_score(y_true=y_test, y_pred=y_pred)

#DataFlair - Print the accuracy
print("Accuracy: {:.2f}%".format(accuracy*100))

## Per-class results (not yet run)

Overall accuracy hides which emotions are confused. This cell prints precision, recall and F1 per emotion and the confusion matrix for the model trained above. It has not been executed in this saved notebook.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

print(classification_report(y_test, y_pred))
print('rows = true emotion, columns = predicted emotion (alphabetical order)')
print(confusion_matrix(y_test, y_pred))

## Speaker-independent check (not yet run)

The split above is random by file, so recordings from the same actor can appear in both train and test, which can inflate the score. This cell holds out whole actors instead. It has not been executed in this saved notebook, so no result is claimed.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=9)
train_idx, test_idx = next(splitter.split(features, labels, groups=actors))

speaker_model = MLPClassifier(alpha=0.01, batch_size=256, epsilon=1e-08, hidden_layer_sizes=(300,),
                              learning_rate='adaptive', max_iter=500, random_state=9)
speaker_model.fit(features[train_idx], labels[train_idx])
speaker_pred = speaker_model.predict(features[test_idx])

print('Actors in the test set:', sorted(set(actors[test_idx])))
print('Speaker-independent accuracy: {:.2f}%'.format(accuracy_score(labels[test_idx], speaker_pred) * 100))